# Module 14 — Tool Calling and Deterministic Tool Execution

Companion notebook to [`docs/modules/14_tool_calling_and_deterministic_execution.md`](../docs/modules/14_tool_calling_and_deterministic_execution.md).

Almost everything here is deterministic Python with no model dependency at all: schema
validation, the registry, permissions, approval gating, real SQLite-backed audit logging, and
budgets. The one LLM-dependent piece is `propose_tool_call()` (asking a model which tool to
call) — `FakeRuntime`-backed here, real adapter unchanged later.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "packages"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_14"))
print(f"Repo root: {REPO_ROOT}")


Repo root: /Users/bhakti/workspace/local_ai_app


## 1. The calculator tool: real AST-based safety, not eval()

In [2]:
from local_ai_agents.tools.calculator import UnsafeExpressionError, safe_eval

print("(2 + 3) * 4 =", safe_eval("(2 + 3) * 4"))
print("2 ** 10 =", safe_eval("2 ** 10"))

for payload in ["__import__('os').system('echo pwned')", "os.system('ls')", "(1).__class__"]:
    try:
        safe_eval(payload)
        print(f"UNSAFE: {payload!r} was evaluated!")
    except UnsafeExpressionError as exc:
        print(f"Rejected: {payload!r} -> {exc}")


(2 + 3) * 4 = 20
2 ** 10 = 1024
Rejected: "__import__('os').system('echo pwned')" -> Disallowed expression element: Call
Rejected: "os.system('ls')" -> Disallowed expression element: Call
Rejected: '(1).__class__' -> Disallowed expression element: Attribute


## 2. Path containment: the same real attack vectors, caught

In [3]:
from local_ai_agents.tools.sandbox import PathTraversalError, resolve_within_sandbox

sandbox = REPO_ROOT / "datasets" / "rag_docs" / "nimbus_handbook"
print("Normal path:", resolve_within_sandbox(sandbox, "password_reset.md").name)

for payload in ["../../../etc/passwd", "/etc/passwd"]:
    try:
        resolve_within_sandbox(sandbox, payload)
        print(f"UNSAFE: {payload!r} resolved inside the sandbox!")
    except PathTraversalError as exc:
        print(f"Rejected: {payload!r} -> {exc}")


Normal path: password_reset.md
Rejected: '../../../etc/passwd' -> '../../../etc/passwd' resolves outside the allowed sandbox /Users/bhakti/workspace/local_ai_app/datasets/rag_docs/nimbus_handbook
Rejected: '/etc/passwd' -> '/etc/passwd' resolves outside the allowed sandbox /Users/bhakti/workspace/local_ai_app/datasets/rag_docs/nimbus_handbook


## 3-4. Tool registry, file search, and the read-only SQL tool — Labs 1-4

In [4]:
from tool_registry_demo import run_lab as run_registry_lab, result_to_markdown as registry_to_markdown

registry_result = await run_registry_lab()
print(registry_to_markdown(registry_result))


# Labs 1-4 - tool registry, calculator, file search, SQL read-only tool

- Registered tools: ['calculator', 'search_files', 'sql_query']
- calculator((2+3)*4): 20
- search_files('15 minutes'): ['password_reset.md']
- sql_query(open tickets): [{'subject': 'Password reset not working'}, {'subject': 'API rate limit hit'}]
- sql_query(DELETE ...) denied: True — Tool execution failed: Only SELECT statements are allowed.
- LLM-proposed tool: calculator -> 144



## 5. SQL tool defense in depth — query-text rejection *and* a real read-only connection

In [5]:
import sqlite3
import tempfile
from pathlib import Path as P

from local_ai_agents.tools.sql_query import UnsafeQueryError, validate_read_only_query

with tempfile.TemporaryDirectory() as tmp:
    db_path = P(tmp) / "demo.db"
    conn = sqlite3.connect(db_path)
    conn.execute("CREATE TABLE t (id INTEGER)")
    conn.execute("INSERT INTO t VALUES (1)")
    conn.commit()
    conn.close()

    try:
        validate_read_only_query("DELETE FROM t")
    except UnsafeQueryError as exc:
        print("Layer 1 (query-text check) rejected DELETE:", exc)

    ro_conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    try:
        ro_conn.execute("DELETE FROM t")
    except sqlite3.OperationalError as exc:
        print("Layer 2 (SQLite's own read-only mode) independently rejected it too:", exc)
    ro_conn.close()


Layer 1 (query-text check) rejected DELETE: Only SELECT statements are allowed.
Layer 2 (SQLite's own read-only mode) independently rejected it too: attempt to write a readonly database


## 6-8. Human approval, permissions, budgets, and real audit logs — Labs 5-6

In [6]:
from approval_and_dangerous_tools_demo import run_lab as run_approval_lab, result_to_markdown as approval_to_markdown

approval_result = await run_approval_lab()
print(approval_to_markdown(approval_result))


# Labs 5-6 - human approval, permissions, budgets, audit logs

- Denied by default (no real approval gate configured): True
- Approved write ('notes.txt', callback approves): True
- Denied write ('secrets.txt', callback denies): True
- Denied by permissions before approval was even asked ('guest' role): True
- Budget test outcomes (3 attempts, 2-call budget): ['success', 'success', 'denied']
- Files actually written to the sandbox: ['notes.txt', 'notes_0.txt', 'notes_1.txt']
- Total audit log entries recorded: 7



## 9. Audit log persistence across a real restart

In [7]:
import tempfile
from pathlib import Path as P

from local_ai_agents.policies.audit_log import AuditLog

with tempfile.TemporaryDirectory() as tmp:
    db_path = P(tmp) / "audit.db"

    log1 = AuditLog(db_path)
    log1.record("trace-1", "calculator", {"expression": "2+2"}, "success", "returned 4")
    log1.close()
    print("Closed log1 (simulating a process restart)...")

    log2 = AuditLog(db_path)  # a genuinely new instance, same file
    entries = log2.entries_for_trace("trace-1")
    print(f"Entries recovered after restart: {len(entries)}")
    for e in entries:
        print(f"  {e.timestamp}  {e.tool_name}  {e.outcome}  {e.detail}")
    log2.close()


Closed log1 (simulating a process restart)...
Entries recovered after restart: 1
  2026-07-09T11:37:46.281418+00:00  calculator  success  returned 4
